# Laboratorio 2 — Ejercicio 2: similitud de series con catch22

**Curso:** CC3084 — Data Science

Se extraen las **22 caracteristicas de catch22** para las siete series construidas en el
Laboratorio 1 (total mensual, tres fronteras, tres vias de ingreso), se construye la matriz
serie x caracteristica, se estandariza y se aplican PCA, clustering jerarquico, un heatmap
de caracteristicas, la matriz de correlacion entre caracteristicas y un mapa de distancias
entre series. Las funciones reutilizables viven en `src/catch22_lab.py`.

## 1. Configuracion

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import catch22_lab as C

print("ROOT:", ROOT)
print("CATCH22_DIR:", C.CATCH22_DIR)

nombres_series = C.cargar_nombres_series()
nombres_series

## 2.1 Idea detras de catch22

catch22 (Lubba et al., 2019, sobre la base de Fulcher, Little y Jones, 2013) resume miles
de caracteristicas posibles de una serie de tiempo (autocorrelacion, entropia, no linealidad,
distribucion de valores, estructura estacional) en solo **22 caracteristicas** que, evaluadas
sobre miles de series reales y sinteticas, maximizan el poder de discriminacion entre series
con la menor redundancia posible y son rapidas de calcular. Es importante porque permite
comparar series de distinta longitud y escala usando un vector corto y estandarizable, en
vez de tener que alinear las series punto a punto.

## 2.2-2.3 Extraccion de las 22 caracteristicas y construccion de la matriz

In [ ]:
matriz, n_observaciones = C.construir_matriz_catch22(nombres_series)
print("Matriz serie x caracteristica:", matriz.shape)
print("Observaciones usadas por serie:", n_observaciones)
matriz

Via Maritima solo tiene observaciones reales entre 2009 y 2016 (dejo de reportarse desde
2017, como se documento en el Laboratorio 1); sus caracteristicas catch22 se calculan sobre
ese tramo valido y contiguo, no sobre el rango completo 2009-2026 de las demas series.

## 2.4 Estandarizacion

In [ ]:
matriz_std, scaler = C.estandarizar_matriz(matriz)
matriz_std.round(2)

## 2.5 PCA

In [ ]:
pca_resultado = C.analisis_pca(matriz_std, C.FIGURAS_DIR / "catch22_pca.png")
print("Varianza explicada (PC1, PC2, ...):", pca_resultado["varianza_explicada"])
pca_resultado["componentes"]

## 2.5 Clustering jerarquico

In [ ]:
clustering_resultado = C.analisis_clustering(matriz_std, C.FIGURAS_DIR / "catch22_dendrograma.png")
print("Silhouette por k:", clustering_resultado["silhouette_por_k"])
clustering_resultado["clusters"]

## 2.5 Heatmap de caracteristicas, correlacion y mapa de distancias

In [ ]:
C.heatmap_caracteristicas(matriz_std, C.FIGURAS_DIR / "catch22_heatmap.png")
corr = C.matriz_correlacion_caracteristicas(matriz_std, C.FIGURAS_DIR / "catch22_correlacion.png")
df_dist = C.mapa_distancias_series(matriz_std, C.FIGURAS_DIR / "catch22_distancias.png")
df_dist

## 2.8 Caracteristicas mas importantes (cargas de PC1 + PC2)

In [ ]:
top_features = C.caracteristicas_mas_importantes(pca_resultado["cargas"])
top_features

## 2.6-2.13 Interpretacion

Ver el texto completo en `resultados/catch22/interpretaciones_catch22.md`. Resumen:

- **Series mas similares (2.7):** Frontera 01 La Aurora y Via Aerea (la distancia mas baja de
  toda la matriz), porque La Aurora concentra casi todo el trafico aereo del pais.
- **Caracteristicas mas discriminantes (2.8):** `DN_HistogramMode_5`, `PD_PeriodicityWang_th0_01`,
  `SB_MotifThree_quantile_hh`, `SC_FluctAnal_2_rsrangefit_50_1_logi_prop_r1` e
  `IN_AutoMutualInfoStats_40_gaussian_fmmi` (mayor peso combinado en PC1+PC2).
- **Grupos naturales (2.9):** el corte optimo por silueta (k=2) solo separa a Via Maritima del
  resto; un corte mas fino (k=3) revela tres grupos: {La Aurora, Via Aerea}, {Total, Valle
  Nuevo, San Cristobal, Via Terrestre} y {Via Maritima}.
- **Cohesion por categoria (2.10):** no es perfecta — Via Aerea se agrupa con una frontera
  (La Aurora) y no con las otras vias; lo que domina el agrupamiento es la composicion real
  del trafico (aereo vs. terrestre), no la etiqueta de categoria.
- **Serie mas atipica (2.11):** Via Maritima, por su cobertura temporal incompleta y su escala
  mucho menor.
- **Consistencia con la Parte 3 del Laboratorio 1 (2.12):** el agrupamiento catch22 es
  consistente con la fuerza de estacionalidad, la pendiente de tendencia y el coeficiente de
  variacion calculados alli (La Aurora y Via Aerea comparten valores casi identicos en las
  tres metricas); el impacto de la pandemia no ayuda a diferenciar porque fue uniformemente
  severo en todas las series.
- **Descubrimientos nuevos (2.13):** ver el punto 3 de `interpretaciones_catch22.md`.

## Guardar todos los resultados (equivalente a `python -m src.catch22_lab`)

In [ ]:
resumen = C.run()
print("Listo. Resultados en resultados/catch22/ y resultados/figuras/.")